# E-Commerce Customer Churn & Retention Analysis
## 03 — Customer Churn Analysis

This notebook examines how customer lifecycle, engagement, purchasing behaviour, and service experience are associated with churn.

### Analytical approach
- `Churn = 1` represents churned customers and `Churn = 0` represents retained customers.
- Churn rates are compared across meaningful customer groups rather than relying only on churn counts.
- Customer counts are reviewed alongside churn rates so that very small groups are not over-interpreted.
- Relationships are treated as **associations**, not evidence of causation.


## 1. Data Preparation

The cleaned customer dataset is loaded for analysis. Data cleaning and validation were completed in the previous notebook, so cleaning steps are not repeated here.


In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("../Data/E Commerce Dataset Cleaned.xlsx")

## 2. Overall Churn Overview

Establish the baseline customer population and overall churn rate before comparing individual customer characteristics.


In [4]:
Total_Customers = df['CustomerID'].count()
Total_Customers

np.int64(5630)

In [5]:
Churned_Customers = df[df['Churn']==1]['CustomerID'].count()
Churned_Customers

np.int64(948)

In [6]:
Retained_Customers = df[df['Churn']==0]['CustomerID'].count()
Retained_Customers

np.int64(4682)

In [7]:
Churn_Rate = ((Churned_Customers/Total_Customers)*100).round(2)
Churn_Rate

np.float64(16.84)

### Baseline Finding

The dataset contains **5,630 customers**, of whom **948 churned** and **4,682 were retained**, producing an overall churn rate of **16.84%**. This rate serves as the benchmark for evaluating higher- and lower-risk customer groups.


## 3. Customer Lifecycle & Engagement

This section evaluates whether customer tenure and engagement characteristics are associated with different churn outcomes.


### 3.1 Tenure

Tenure is grouped into meaningful lifecycle stages to compare churn among newer and longer-standing customers.


In [8]:
df["TenureGroup"] = pd.cut(
    df["Tenure"],
    bins=[-1, 3, 6, 12, 24, float("inf")],
    labels=["0-3 Months", "4-6 Months", "7-12 Months", "13-24 Months", "25+ Months"]
)

In [10]:
df.groupby("TenureGroup")["CustomerID"].count()

TenureGroup
0-3 Months      1560
4-6 Months       590
7-12 Months     1584
13-24 Months    1467
25+ Months       429
Name: CustomerID, dtype: int64

In [24]:
(df.groupby("TenureGroup")["Churn"].mean()*100).round(2)

TenureGroup
0-3 Months      41.86
4-6 Months       7.46
7-12 Months      9.85
13-24 Months     6.48
25+ Months       0.00
Name: Churn, dtype: float64

**Finding:** Churn is heavily concentrated among customers in their first three months. The **0–3 month** segment has a **41.86% churn rate**, substantially above every longer-tenure segment. This identifies the early customer lifecycle as a critical retention period. The pattern should not be described as a continuous decline because churn does not decrease monotonically across all later tenure groups.


### 3.2 Time Spent on App

App usage is compared with churn to determine whether time spent on the platform provides a useful engagement signal.


In [19]:
df.groupby("HourSpendOnApp")['CustomerID'].count()

HourSpendOnApp
0       3
1      35
2    1471
3    2942
4    1176
5       3
Name: CustomerID, dtype: int64

In [23]:
(df.groupby("HourSpendOnApp")["Churn"].mean()*100).round(2)

HourSpendOnApp
0     0.00
1     0.00
2    15.77
3    17.61
4    16.84
5     0.00
Name: Churn, dtype: float64

**Finding:** App usage hours show **no strong standalone relationship with churn**. The major 2–4 hour groups have similar churn rates of roughly 16–18%. Extremely small groups at 0, 1, and 5 hours are not suitable for meaningful inference.


### 3.3 Number of Registered Devices

The analysis compares churn across the number of devices registered by each customer.


In [25]:
df.groupby("NumberOfDeviceRegistered")["CustomerID"].count()

NumberOfDeviceRegistered
1     235
2     276
3    1699
4    2377
5     881
6     162
Name: CustomerID, dtype: int64

In [29]:
(df.groupby("NumberOfDeviceRegistered")['Churn'].mean()*100).round(2)

NumberOfDeviceRegistered
1     9.36
2     9.42
3    14.95
4    16.49
5    22.47
6    34.57
Name: Churn, dtype: float64

**Finding:** The number of registered devices shows a **strong positive association with churn**. Churn rises from about **9% for customers with 1–2 registered devices** to **22.47% for customers with 5 devices**. The 6-device group shows an even higher rate but contains fewer customers, so its exact rate should be interpreted cautiously.


### 3.4 Preferred Login Device

Churn is compared between customers who primarily log in using a phone and those who prefer a computer.


In [30]:
df.groupby("PreferredLoginDevice")['CustomerID'].count()

PreferredLoginDevice
Computer    1634
Phone       3996
Name: CustomerID, dtype: int64

In [32]:
(df.groupby("PreferredLoginDevice")['Churn'].mean()*100).round(2)

PreferredLoginDevice
Computer    19.83
Phone       15.62
Name: Churn, dtype: float64

**Finding:** Customers who prefer **computers** show moderately higher churn than **phone** users (**19.83% vs 15.62%**). This is a secondary churn signal rather than one of the strongest relationships in the analysis.


## 4. Purchasing Behaviour

This section examines whether order frequency, purchase recency, and preferred product category distinguish customers with different churn behaviour.


### 4.1 Order Frequency

Individual order counts become fragmented at higher values, so customers are grouped into broader order-frequency bands.


In [40]:
df["OrderCountGroup"] = pd.cut(
    df['OrderCount'],
    bins=[0, 2, 5, 10, float("inf")],
    labels=["1-2 Orders", "3-5 Orders", "6-10 Orders", "11+ Order"]
)

In [41]:
df.groupby("OrderCountGroup")['CustomerID'].count()

OrderCountGroup
1-2 Orders     4034
3-5 Orders      756
6-10 Orders     613
11+ Order       227
Name: CustomerID, dtype: int64

In [43]:
(df.groupby("OrderCountGroup")["Churn"].mean()*100).round(2)

OrderCountGroup
1-2 Orders     17.45
3-5 Orders     14.55
6-10 Orders    17.29
11+ Order      12.33
Name: Churn, dtype: float64

**Finding:** Order frequency does **not show a strong or consistent standalone relationship with churn**. The 11+ order group has lower churn, but churn does not steadily decrease as order frequency increases across all groups.


### 4.2 Days Since Last Order

Purchase recency is grouped into time bands to evaluate whether the time elapsed since a customer's last order is associated with churn.


In [45]:
df["DaySinceLastOrderGroup"] = pd.cut(
    df["DaySinceLastOrder"],
    bins=[-1,3,7,14,30,float("inf")],
    labels=['0-3 Days','4-7 Days','8-14 Days', '15-30 Days', '31+ Days']
)

In [46]:
df.groupby("DaySinceLastOrderGroup")['CustomerID'].count()

DaySinceLastOrderGroup
0-3 Days      3109
4-7 Days      1219
8-14 Days     1240
15-30 Days      60
31+ Days         2
Name: CustomerID, dtype: int64

In [48]:
(df.groupby("DaySinceLastOrderGroup")['Churn'].mean()*100).round(2)

DaySinceLastOrderGroup
0-3 Days      21.16
4-7 Days      13.70
8-14 Days      9.52
15-30 Days     6.67
31+ Days      50.00
Name: Churn, dtype: float64

**Finding:** Purchase recency shows an **unexpected relationship** with churn. Customers who ordered within the last **0–3 days** have the highest meaningful-group churn rate at **21.16%**, while churn is lower among customers whose last order was 4–14 days ago. The 15–30 and 31+ day groups are very small and should not drive conclusions.


### 4.3 Preferred Order Category

Customer counts and churn rates are compared across preferred product categories.


In [52]:
df.groupby("PreferedOrderCat")['CustomerID'].count()

PreferedOrderCat
Fashion                826
Grocery                410
Laptop & Accessory    2050
Mobile                2080
Others                 264
Name: CustomerID, dtype: int64

In [54]:
(df.groupby("PreferedOrderCat")['Churn'].mean()*100).round(2)

PreferedOrderCat
Fashion               15.50
Grocery                4.88
Laptop & Accessory    10.24
Mobile                27.40
Others                 7.58
Name: Churn, dtype: float64

**Finding:** Preferred order category shows a **strong association with churn**. Customers who prefer **Mobile** products have the highest churn rate at **27.40%**, well above the 16.84% overall rate. **Grocery** customers have the lowest churn rate at **4.88%**.


## 5. Customer Experience

Customer experience is evaluated through satisfaction scores and complaint history.


### 5.1 Satisfaction Score

Churn rates are compared across the five satisfaction-score levels.


In [56]:
df.groupby('SatisfactionScore')['CustomerID'].count()

SatisfactionScore
1    1164
2     586
3    1698
4    1074
5    1108
Name: CustomerID, dtype: int64

In [58]:
(df.groupby('SatisfactionScore')['Churn'].mean()*100).round(2)

SatisfactionScore
1    11.51
2    12.63
3    17.20
4    17.13
5    23.83
Name: Churn, dtype: float64

**Finding:** Satisfaction score shows a meaningful but **counterintuitive association** with churn. Customers with a score of **5** have the highest churn rate at **23.83%**, compared with **11.51%** for customers with a score of 1. Because the observed direction differs from the conventional expectation for a satisfaction measure, the relationship is reported without assigning a causal explanation.


### 5.2 Complaint History

Complaint status is compared with churn to assess whether customers who have registered a complaint exhibit materially different retention outcomes.


In [60]:
df.groupby("Complain")['CustomerID'].count()

Complain
0    4026
1    1604
Name: CustomerID, dtype: int64

In [62]:
(df.groupby("Complain")['Churn'].mean()*100).round(2)

Complain
0    10.93
1    31.67
Name: Churn, dtype: float64

**Finding:** Complaint history is one of the **strongest churn indicators** identified. Customers who registered a complaint have a **31.67% churn rate**, compared with **10.93%** among customers without a complaint.


## 6. Key Findings

The analysis identifies several customer characteristics that are meaningfully associated with churn:

- **Early tenure is the strongest lifecycle risk signal:** customers in their first 0–3 months have a 41.86% churn rate.
- **Complaints are strongly associated with churn:** customers with a complaint churn at 31.67%, versus 10.93% for those without one.
- **Preferred order category matters:** Mobile-preferring customers show 27.40% churn, substantially above the overall rate.
- **More registered devices are associated with higher churn**, particularly among customers with five or more devices.
- **Computer-preferring customers show moderately higher churn** than phone-preferring customers.
- **Purchase recency and satisfaction score show meaningful but counterintuitive patterns** and should therefore be interpreted cautiously.
- **Time spent on the app and order frequency do not provide strong standalone churn signals.**

These findings provide the evidence base for the next stage: combining the strongest factors into meaningful customer risk segments and retention priorities.
